# UR5 equations of motion — three forward-dynamics pipelines

Textbook-style comparison of how minilink evaluates UR5 joint accelerations $\ddot q$ from

$$
H(q)\,\ddot q + C(q,\dot q)\,\dot q + d(q,\dot q) + g(q) = \tau.
$$

We study **three pipelines** that target the same $\ddot q$ but differ in algorithm and cost:

| Pipeline | Idea | minilink entry |
| --- | --- | --- |
| **RNEA–$H$** | RNEA bias + explicit inertia solve | `UR5Manipulator.forward_dynamics` |
| **ABA** | Articulated-body algorithm ($O(n)$ spatial) | `UR5Manipulator.forward_dynamics_aba` |
| **Symbolic Lagrange** | Derive $H,C,g$ once; lambdify | `minilink.symbolic` → `to_minilink()` |

Sections **6–7** compare single-step forward dynamics; section **8** integrates a full rollout with separate **compile**, **JIT warm-up**, and **integration** timings.

Helpers: `compare_eom.py`, `rollout_compare.py`, `symbolic_ur5.py`. Thin CLI smoke: `run_demo.py`.


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

repo = Path.cwd()
if not (repo / "minilink").is_dir():
    repo = repo.parents[2]
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from examples.projects.ur5_dynamics.compare_eom import (
    DEFAULT_N_SAMPLES,
    DEFAULT_N_TIMING,
    DEFAULT_SEED,
    aba_speedup,
    accuracy_summary,
    build_evaluation_batch,
    comprehensive_comparison,
    forward_dynamics_aba,
    forward_dynamics_rnea_h,
    forward_dynamics_symbolic,
    named_case_summary,
    plot_comparison,
    plot_per_joint_errors,
)
from examples.projects.ur5_dynamics.rollout_compare import (
    DEFAULT_ROLLOUT_BACKEND,
    DEFAULT_ROLLOUT_DT,
    DEFAULT_ROLLOUT_TF,
    plot_rollout_timings,
    rollout_comparison,
    rollout_error_rows,
    rollout_timing_rows,
)
from examples.projects.ur5_dynamics.symbolic_ur5 import (
    build_symbolic_ur5,
    catalog_params_no_damping,
)
from minilink.dynamics.catalog.manipulators.ur5 import UR5Manipulator

%matplotlib inline

## 1. Problem statement

For an $n$-link serial manipulator,

$$
H(q)\,\ddot q + b(q,\dot q) + d(q,\dot q) = \tau,
$$

with $H \succ 0$, bias $b$ (Coriolis, centrifugal, gravity), dissipation $d$, and torque $\tau$.
Forward dynamics solves

$$
\ddot q = H(q)^{-1}\bigl(\tau - b(q,\dot q) - d(q,\dot q)\bigr).
$$

The UR5 catalog plant uses spatial RNEA internally. Below we **disable viscous damping** so the symbolic Lagrange export matches the spatial parameters.


## 2. Pipeline A — RNEA bias + explicit $H$

1. Bias force: $b(q,\dot q) = \mathrm{RNEA}(q,\dot q, 0)$ — one $O(n)$ tree pass.
2. Inertia columns: $H_{:,j} = \mathrm{RNEA}(q, 0, e_j)$ for $j = 1,\ldots,n$ — $n$ passes $\Rightarrow O(n^2)$.
3. Solve $\ddot q = H^{-1}(\tau - b - d)$ — $O(n^3)$; negligible for UR5 ($n=6$).

**Per call:** $O(n^2)$ dominated by forming $H$. **Strength:** exposes $H$, $g$, $C$ for control design.


## 3. Pipeline B — Articulated Body Algorithm (ABA)

Three spatial passes without assembling $H$:

1. Outward: link velocities $v_i$, Coriolis terms $c_i$, bias $p_{A,i}$.
2. Inward: articulated inertias $I_i^A$, scalars $U_i$, $d_i$, $u_i$.
3. Outward: spatial accelerations and joint $\ddot q_i$.

**Per call:** $O(n)$. **Strength:** best scaling for long-chain simulation.


## 4. Pipeline C — Symbolic Lagrange

Build a DH chain in SymPy, form $T(q,\dot q) - V(q)$, and apply Euler–Lagrange. Export once via `to_minilink()`.

| Phase | Cost |
| --- | --- |
| Symbolic derive (once) | minutes for UR5 |
| Lambdify export (once) | similar |
| Each FD call | $O(n^2)$ + solve — like RNEA–$H$ |

**Strength:** automatic model generation and verification.


## 5. Hands-on — one configuration

Before the batch study, call each pipeline on the same $(q, \dot q, \tau)$.


In [ ]:
arm = UR5Manipulator()
params = catalog_params_no_damping()

q = np.array([0.1, -0.5, 0.2, -1.0, 0.3, 0.0])
v = np.array([0.5, -0.3, 0.2, 0.1, -0.4, 0.2])
u = np.zeros(6)

qdd_rnea = forward_dynamics_rnea_h(arm, q, v, u, params)
qdd_aba = forward_dynamics_aba(arm, q, v, u, params)

print("RNEA–H qdd:", np.array2string(qdd_rnea, precision=4, suppress_small=True))
print("ABA    qdd:", np.array2string(qdd_aba, precision=4, suppress_small=True))
print("max |Δqdd| (ABA vs RNEA–H):", np.max(np.abs(qdd_rnea - qdd_aba)))

In [ ]:
# Symbolic plant: first call ~2–3 min; cached afterward.
# Set os.environ["UR5_SKIP_SYMBOLIC"] = "1" before this cell for an ABA-only run.
symbolic_plant = None
if os.environ.get("UR5_SKIP_SYMBOLIC", "").strip().lower() not in {"1", "true", "yes"}:
    try:
        symbolic_plant = build_symbolic_ur5(verbose=True)
        qdd_sym = forward_dynamics_symbolic(symbolic_plant, q, v, u)
        print("Symbolic qdd:", np.array2string(qdd_sym, precision=4, suppress_small=True))
        print("max |Δqdd| (Symbolic vs RNEA–H):", np.max(np.abs(qdd_rnea - qdd_sym)))
    except ImportError:
        print("SymPy not installed — pip install minilink[symbolic]")
else:
    print("UR5_SKIP_SYMBOLIC set — skipping symbolic pipeline.")

## 6. Single-step batch study — accuracy and timing

**67 samples:** three named poses plus **64 random** draws with **seed 0** (reproducible). Reference: RNEA–$H$.


In [ ]:
SEED = DEFAULT_SEED
N_RANDOM = DEFAULT_N_SAMPLES
N_TIMING = DEFAULT_N_TIMING

configs, labels = build_evaluation_batch(seed=SEED, n_random=N_RANDOM)
result = comprehensive_comparison(
    arm,
    symbolic_plant,
    configs,
    params=params,
    seed=SEED,
    n_timing=N_TIMING,
    case_labels=labels,
)

print(f"Batch: {result.n_samples} samples ({N_RANDOM} random + 3 named, seed={SEED})")

In [ ]:
def _rows_to_markdown(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        cells = []
        for col in columns:
            val = row[col]
            if isinstance(val, float):
                if col.endswith("_ms"):
                    cells.append(f"{val:.3f}")
                elif val == 0.0:
                    cells.append("0")
                else:
                    cells.append(f"{val:.3e}")
            else:
                cells.append(str(val))
        body.append("| " + " | ".join(cells) + " |")
    return "\n".join([header, sep] + body)

acc_cols = [
    "method",
    "max_abs_dqdd",
    "mean_abs_dqdd",
    "rms_abs_dqdd",
    "p95_abs_dqdd",
    "median_ms",
]
display(Markdown("### Accuracy and timing (vs RNEA–H reference)"))
display(Markdown(_rows_to_markdown(accuracy_summary(result), acc_cols)))

case_cols = list(named_case_summary(result)[0].keys())
display(Markdown("### Named case studies"))
display(Markdown(_rows_to_markdown(named_case_summary(result), case_cols)))

display(Markdown(f"**ABA median speedup vs RNEA–H:** {aba_speedup(result):.2f}×"))

In [ ]:
fig, _ = plot_comparison(result)
fig.suptitle("UR5 forward dynamics — random-batch accuracy and timing", y=1.02)
plt.show()

In [ ]:
if "Symbolic" in result.methods:
    fig2, _ = plot_per_joint_errors(result)
    plt.show()
else:
    print("Per-joint symbolic plot skipped (symbolic plant not built).")

## 8. Rollout simulation — trajectory parity and split timings

Integrate $\dot x = f(x,u)$ with fixed-step RK4 (`compile_backend="jax"`, `tf=1` s, `dt=2` ms, zero torque).

We report three wall-clock phases per pipeline:

| Phase | What it measures |
| --- | --- |
| **compile** | `Simulator` construction (graph build) |
| **JIT warm-up** | first `solve()` — JAX compilation of the RK4 scan + first integration |
| **rollout** | second `solve()` — full fixed-step integration only |

Reference trajectory: **RNEA–$H$**. Compare $\max |x - x_\mathrm{ref}|$ over the grid for ABA and symbolic paths.

In [ ]:
# JAX-exported symbolic plant for rollout (cached separately from NumPy FD plant).
symbolic_rollout = None
if symbolic_plant is not None:
    symbolic_rollout = build_symbolic_ur5(backend=DEFAULT_ROLLOUT_BACKEND, verbose=False)

rollout_result, ref_traj = rollout_comparison(
    params,
    symbolic_plant=symbolic_rollout,
    compile_backend=DEFAULT_ROLLOUT_BACKEND,
    tf=DEFAULT_ROLLOUT_TF,
    dt=DEFAULT_ROLLOUT_DT,
)
print(
    f"Rollout grid: tf={rollout_result.tf}s, dt={rollout_result.dt}s, "
    f"n={rollout_result.n_steps}, backend={rollout_result.backend}"
)

In [ ]:
timing_cols = ["method", "compile_ms", "jit_warmup_ms", "rollout_ms", "backend"]
display(Markdown("### Rollout timings"))
display(Markdown(_rows_to_markdown(rollout_timing_rows(rollout_result), timing_cols)))

if rollout_result.max_state_error:
    err_cols = list(rollout_error_rows(rollout_result)[0].keys())
    display(Markdown("### Trajectory error vs RNEA–H"))
    display(Markdown(_rows_to_markdown(rollout_error_rows(rollout_result), err_cols)))

In [ ]:
fig3, _ = plot_rollout_timings(rollout_result)
plt.show()

## 9. Summary

* **Single-step FD:** ABA and symbolic Lagrange match RNEA–$H$ to near machine precision on the full batch ($\max |\Delta \ddot q| \lesssim 10^{-10}$ rad/s$^2$ typically).
* **Rollout:** integrated trajectories agree to similar tolerance when JAX uses float64; **JIT warm-up** is paid once, then **rollout** timing reflects pure integration.
* **Speed:** ABA wins both per FD call and per rollout step count because it never forms $H$.

**When to use which:** ABA for simulation loops on long chains; RNEA–$H$ (or exported symbolic $H$, $g$) for teaching, linearization, and control design.
